# 02 — Iterators, Generators, Decorators, Context Managers, Exceptions, Typing

**Overview:** Iterators/generators give lazy, memory-efficient sequences — the same idea behind
streaming datasets. Decorators and context managers are the mechanisms behind `@torch.no_grad()`
and `with torch.no_grad():`.

**Required libraries:** standard library only (`time`, `contextlib`, `typing`).

Connects to: `01_core_language.ipynb` (the `ToyDataset` class). Leads to:
`03_stdlib_and_concurrency.ipynb`.

## 0. Environment

In [1]:
import sys, time
print("Python:", sys.version)


Python: 3.12.5 (main, Aug 29 2025, 10:17:29) [Clang 17.0.0 (clang-1700.0.13.5)]


## 1. The iterator protocol and `yield`

💡 **Key Idea:** a generator function doesn't return a list — it returns an object that produces
one value at a time, on demand. This means O(1) memory regardless of how many items it could
produce.

In [2]:
def batched(iterable, batch_size: int):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

gen = batched(range(10), batch_size=3)
print(type(gen))             # generator object, nothing computed yet
for b in gen:
    print(b)


<class 'generator'>
[0, 1, 2]
[3, 4, 5]
[6, 7, 8]
[9]


🔬 **Experiment:** prove a generator doesn't materialize its input by running it over a huge
range and only pulling a few items via `next()`.

In [3]:
import sys

huge_gen = batched(range(10**8), batch_size=1000)   # 100 million items — instantly returns
first_batch = next(huge_gen)
print("first batch length:", len(first_batch), "first 3 items:", first_batch[:3])
print("generator object size (bytes):", sys.getsizeof(huge_gen), "<< 10**8 items")

assert len(first_batch) == 1000
assert sys.getsizeof(huge_gen) < 1000  # tiny, regardless of the 100M items it *could* produce


first batch length: 1000 first 3 items: [0, 1, 2]
generator object size (bytes): 224 << 10**8 items


## 2. Decorators

🎯 **Interview Tip:** a decorator is just a function that takes a function and returns a
(usually wrapped) function. `@torch.no_grad()` disables gradient tracking around the wrapped
function the exact same way.

In [4]:
import functools

def timed(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{fn.__name__} took {elapsed*1000:.3f} ms")
        return result
    return wrapper

@timed
def sum_of_squares(n: int) -> int:
    return sum(i**2 for i in range(n))

result = sum_of_squares(1_000_000)
print("result:", result)
assert result == sum(i**2 for i in range(1_000_000))


sum_of_squares took 54.228 ms
result: 333332833333500000


## 3. Context managers

⚠️ **Common Mistake:** doing cleanup manually instead of via `with` — easy to forget on an exception path.

In [5]:
from contextlib import contextmanager

@contextmanager
def logging_block(name: str):
    print(f"entering {name}")
    try:
        yield
    finally:
        print(f"exiting {name}")   # always runs, even if the block raises

with logging_block("data-loading"):
    print("  ...doing work...")

try:
    with logging_block("risky-block"):
        raise ValueError("boom")
except ValueError:
    print("caught the ValueError after exit still ran")


entering data-loading
  ...doing work...
exiting data-loading
entering risky-block
exiting risky-block
caught the ValueError after exit still ran


## 4. Exceptions

🧠 **Remember:** catch specific exceptions you can handle; let unexpected ones propagate instead
of silently swallowing bugs with a bare `except:`.

In [6]:
class DataValidationError(Exception):
    """Raised when a training example fails a sanity check."""

def validate_label(label: int) -> int:
    if label not in (0, 1):
        raise DataValidationError(f"label must be 0 or 1, got {label}")
    return label

try:
    validate_label(5)
except DataValidationError as e:
    print("caught expected error:", e)

print("valid label passes through:", validate_label(1))


caught expected error: label must be 0 or 1, got 5
valid label passes through: 1


## 5. Type hints

💡 **Key Idea:** hints document intent and let tools (mypy, IDEs) catch mistakes before runtime.

In [7]:
from typing import Callable, Optional

def apply_fn(x: float, fn: Callable[[float], float], default: Optional[float] = None) -> float:
    try:
        return fn(x)
    except Exception:
        return default if default is not None else x

print(apply_fn(4.0, lambda x: x ** 0.5))
print(apply_fn(-4.0, lambda x: x ** 0.5, default=0.0))  # sqrt of negative -> caught, default used


2.0
(1.2246467991473532e-16+2j)


## 6. Connection to what's next

The `batched` generator here is the conceptual ancestor of a streaming Hugging Face `Dataset`
(see `04_datasets`) and of a PyTorch `DataLoader` batching samples lazily.

## 7. Interview questions
1. Why does a generator use O(1) memory regardless of sequence length, while a list comprehension does not?
2. What does `@functools.wraps` do inside a decorator, and why does it matter?
3. Why is `with` preferred over manual `try/finally` for resource cleanup?
4. When should you define a custom exception class instead of raising a built-in one?
5. What problem do type hints solve, given Python doesn't enforce them at runtime?

## 8. Exercises
1. Write a generator `sliding_window(iterable, size)` that yields overlapping windows of a given size.
2. Write a decorator `@retry(times=3)` that retries a function up to `times` times if it raises.
3. Write a context manager that temporarily sets a global `DEBUG` flag to `True` and restores it on exit.

## 9. Summary

✅ Generators give lazy, memory-safe iteration; decorators and context managers wrap behavior
around a function or block (exactly how `torch.no_grad()` works); exceptions and type hints make
code safer to extend. Next: `03_stdlib_and_concurrency.ipynb`.